## Kaggle Setup — 1. Preprocessing
### Datasets to add (Notebook → Add Data):
- **m5-raw** : Upload `calendar.csv`, `sell_prices.csv`, `sales_train_evaluation.csv`, `sales_train_validation.csv`, `sample_submission.csv`

### Output:
- Saves `grid_part_1.pkl`, `grid_part_2.pkl`, `grid_part_3.pkl`, `lags_df_28.pkl`, `mean_encoding_df.pkl` to `/kaggle/working/processed/`
- After running, download and upload them as a **new dataset `m5-processed`** to reuse across other notebooks.

> **Accelerator:** CPU is fine. Expected time: ~1.5–2 hrs on Kaggle.


In [ ]:
raw_data_dir       = '../'
processed_data_dir = './processed/'

import os
os.makedirs(processed_data_dir, exist_ok=True)


# 1. Main setup

In [ ]:
# General imports
import numpy as np
import pandas as pd
import os, sys, gc, time, warnings, pickle, psutil, random

from math import ceil

from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

warnings.filterwarnings('ignore')

In [ ]:
## Simple "Memory profilers" to see memory usage
def get_memory_usage():
    return np.round(psutil.Process(os.getpid()).memory_info()[0]/2.**30, 2) 
        
def sizeof_fmt(num, suffix='B'):
    for unit in ['','Ki','Mi','Gi','Ti','Pi','Ei','Zi']:
        if abs(num) < 1024.0:
            return "%3.1f%s%s" % (num, unit, suffix)
        num /= 1024.0
    return "%.1f%s%s" % (num, 'Yi', suffix)

In [ ]:
## Memory Reducer
# :df pandas dataframe to reduce size             # type: pd.DataFrame()
# :verbose                                        # type: bool
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2    
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                       df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

In [ ]:
## Merging by concat to not lose dtypes
def merge_by_concat(df1, df2, merge_on):
    merged_gf = df1[merge_on]
    merged_gf = merged_gf.merge(df2, on=merge_on, how='left')
    new_columns = [col for col in list(merged_gf) if col not in merge_on]
    df1 = pd.concat([df1, merged_gf[new_columns]], axis=1)
    return df1

In [ ]:
########################### Vars
#################################################################################
TARGET = 'sales'         # Our main target
END_TRAIN = 1941         # Last day in train set
MAIN_INDEX = ['id','d']  # We can identify item by these columns
STORES = ['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']


# 2. Part 1
- Melting train data => grid_part_1
- creating price features => grid_part_2
- creating calendar features => grid_part_3

In [ ]:
########################### Load Data
#################################################################################
print('Load Main Data')

# Here are reafing all our data 
# without any limitations and dtype modification
train_df = pd.read_csv(raw_data_dir+'sales_train_evaluation.csv')
prices_df = pd.read_csv(raw_data_dir+'sell_prices.csv')
calendar_df = pd.read_csv(raw_data_dir+'calendar.csv')

In [ ]:
########################### STORE LOOP PROCESSING
#################################################################################
print('Starting Store-by-Store Processing...')

# Load global prices and calendar once
prices_df_global = pd.read_csv(raw_data_dir+'sell_prices.csv')
calendar_df_global = pd.read_csv(raw_data_dir+'calendar.csv')

# Load train_df once
train_df_global = pd.read_csv(raw_data_dir+'sales_train_evaluation.csv')

for store in STORES:
    print(f'\n================ {store} ================')
    
    # Filter raw data for this store BEFORE melting
    train_df = train_df_global[train_df_global['store_id'] == store].copy()
    prices_df = prices_df_global[prices_df_global['store_id'] == store].copy()
    calendar_df = calendar_df_global.copy()
    
    # -------------------------------------------------------------
    # 1. MAKE GRID
    # -------------------------------------------------------------
    print('Create Grid')
    index_columns = ['id','item_id','dept_id','cat_id','store_id','state_id']
    grid_df = pd.melt(train_df, 
                      id_vars = index_columns, 
                      var_name = 'd', 
                      value_name = TARGET)

    add_grid = pd.DataFrame()
    for i in range(1,29):
        temp_df = train_df[index_columns]
        temp_df = temp_df.drop_duplicates()
        temp_df['d'] = 'd_'+ str(END_TRAIN+i)
        temp_df[TARGET] = np.nan
        add_grid = pd.concat([add_grid,temp_df])

    grid_df = pd.concat([grid_df,add_grid])
    grid_df = grid_df.reset_index(drop=True)
    del temp_df, add_grid, train_df
    
    for col in index_columns:
        grid_df[col] = grid_df[col].astype('category')
        
    # -------------------------------------------------------------
    # Product Release date
    # -------------------------------------------------------------
    release_df = prices_df.groupby(['store_id','item_id'])['wm_yr_wk'].agg(['min']).reset_index()
    release_df.columns = ['store_id','item_id','release']

    grid_df = merge_by_concat(grid_df, release_df, ['store_id','item_id'])
    del release_df

    grid_df = merge_by_concat(grid_df, calendar_df[['wm_yr_wk','d']], ['d'])
                          
    grid_df = grid_df[grid_df['wm_yr_wk']>=grid_df['release']]
    grid_df = grid_df.reset_index(drop=True)

    grid_df['release'] = grid_df['release'] - grid_df['release'].min()
    grid_df['release'] = grid_df['release'].astype(np.int16)
    
    # Convert 'd' to int and remove 'wm_yr_wk' (from additional cleaning cell)
    grid_df['d'] = grid_df['d'].apply(lambda x: x[2:]).astype(np.int16)
    del grid_df['wm_yr_wk']

    print('Save Part 1')
    grid_df.to_pickle(processed_data_dir+f'grid_part_1_{store}.pkl')
    
    # -------------------------------------------------------------
    # 2. PRICES
    # -------------------------------------------------------------
    print('Prices')
    prices_df['price_max'] = prices_df.groupby(['store_id','item_id'])['sell_price'].transform('max')
    prices_df['price_min'] = prices_df.groupby(['store_id','item_id'])['sell_price'].transform('min')
    prices_df['price_std'] = prices_df.groupby(['store_id','item_id'])['sell_price'].transform('std')
    prices_df['price_mean'] = prices_df.groupby(['store_id','item_id'])['sell_price'].transform('mean')
    prices_df['price_norm'] = prices_df['sell_price']/prices_df['price_max']

    prices_df['price_nunique'] = prices_df.groupby(['store_id','item_id'])['sell_price'].transform('nunique') 
    prices_df['item_nunique'] = prices_df.groupby(['store_id','sell_price'])['item_id'].transform('nunique')

    calendar_prices = calendar_df[['wm_yr_wk','month','year']].drop_duplicates(subset=['wm_yr_wk'])
    prices_df = prices_df.merge(calendar_prices[['wm_yr_wk','month','year']], on=['wm_yr_wk'], how='left')
    del calendar_prices

    prices_df['price_momentum'] = prices_df['sell_price']/prices_df.groupby(['store_id','item_id'])['sell_price'].transform(lambda x: x.shift(1))
    prices_df['price_momentum_m'] = prices_df['sell_price']/prices_df.groupby(['store_id','item_id','month'])['sell_price'].transform('mean')
    prices_df['price_momentum_y'] = prices_df['sell_price']/prices_df.groupby(['store_id','item_id','year'])['sell_price'].transform('mean')
    del prices_df['month'], prices_df['year']
    
    grid_df = reduce_mem_usage(grid_df)
    prices_df = reduce_mem_usage(prices_df)
    
    # Need to merge with wm_yr_wk temporarily
    calendar_temp = calendar_df[['wm_yr_wk', 'd']].copy()
    calendar_temp['d_int'] = calendar_temp['d'].apply(lambda x: int(x[2:]))
    grid_df = grid_df.merge(calendar_temp, left_on='d', right_on='d_int', how='left')
    grid_df = grid_df.drop(['d_y', 'd_int'], axis=1).rename(columns={'d_x': 'd'})
    
    original_columns = list(grid_df)
    grid_df = grid_df.merge(prices_df, on=['store_id','item_id','wm_yr_wk'], how='left')
    keep_columns = [col for col in list(grid_df) if col not in original_columns]
    
    grid_part_2 = grid_df[MAIN_INDEX+keep_columns]
    grid_part_2 = reduce_mem_usage(grid_part_2)
    
    print('Save Part 2')
    grid_part_2.to_pickle(processed_data_dir+f'grid_part_2_{store}.pkl')
    del grid_part_2, prices_df
    
    del grid_df['wm_yr_wk']
    
    # -------------------------------------------------------------
    # 3. CALENDAR
    # -------------------------------------------------------------
    print('Calendar')
    grid_part_3 = grid_df[MAIN_INDEX].copy()
    
    icols = ['date','d','event_name_1','event_type_1','event_name_2','event_type_2','snap_CA','snap_TX','snap_WI']
    # Restore string 'd' for merging
    calendar_df['d_int'] = calendar_df['d'].apply(lambda x: int(x[2:]))
    grid_part_3 = grid_part_3.merge(calendar_df[icols + ['d_int']], left_on='d', right_on='d_int', how='left')
    
    icols_cat = ['event_name_1','event_type_1','event_name_2','event_type_2','snap_CA','snap_TX','snap_WI']
    for col in icols_cat:
        grid_part_3[col] = grid_part_3[col].astype('category')
        
    grid_part_3['date'] = pd.to_datetime(grid_part_3['date'])
    grid_part_3['tm_d'] = grid_part_3['date'].dt.day.astype(np.int8)
    grid_part_3['tm_w'] = grid_part_3['date'].dt.isocalendar().week.astype(np.int8)
    grid_part_3['tm_m'] = grid_part_3['date'].dt.month.astype(np.int8)
    grid_part_3['tm_y'] = grid_part_3['date'].dt.year
    grid_part_3['tm_y'] = (grid_part_3['tm_y'] - grid_part_3['tm_y'].min()).astype(np.int8)
    grid_part_3['tm_wm'] = grid_part_3['tm_d'].apply(lambda x: ceil(x/7)).astype(np.int8)
    grid_part_3['tm_dw'] = grid_part_3['date'].dt.dayofweek.astype(np.int8) 
    grid_part_3['tm_w_end'] = (grid_part_3['tm_dw']>=5).astype(np.int8)
    del grid_part_3['date'], grid_part_3['d_y'], grid_part_3['d_int']
    grid_part_3 = grid_part_3.rename(columns={'d_x': 'd'})
    
    print('Save Part 3')
    grid_part_3 = grid_part_3.drop(['id'], axis=1)
    grid_part_3['id'] = grid_df['id']
    grid_part_3['d'] = grid_df['d']
    grid_part_3.to_pickle(processed_data_dir+f'grid_part_3_{store}.pkl')
    del grid_part_3
    
    # -------------------------------------------------------------
    # 4. LAGS & ROLLINGS
    # -------------------------------------------------------------
    print('Lags & Rollings')
    grid_df['sales'][grid_df['d']>(1941-28)] = np.nan
    SHIFT_DAY = 28
    
    lag_df = grid_df[['id','d','sales']].copy()
    LAG_DAYS = [col for col in range(SHIFT_DAY,SHIFT_DAY+15)]
    lag_df = lag_df.assign(**{
            '{}_lag_{}'.format(col, l): lag_df.groupby(['id'])[col].transform(lambda x: x.shift(l))
            for l in LAG_DAYS
            for col in [TARGET]
        })
        
    for col in list(lag_df):
        if 'lag' in col:
            lag_df[col] = lag_df[col].astype(np.float16)
            
    for i in [7,14,30,60,180]:
        lag_df['rolling_mean_'+str(i)] = lag_df.groupby(['id'])[TARGET].transform(lambda x: x.shift(SHIFT_DAY).rolling(i).mean()).astype(np.float16)
        lag_df['rolling_std_'+str(i)]  = lag_df.groupby(['id'])[TARGET].transform(lambda x: x.shift(SHIFT_DAY).rolling(i).std()).astype(np.float16)
        
    for d_shift in [1,7,14]: 
        for d_window in [7,14,30,60]:
            col_name = 'rolling_mean_tmp_'+str(d_shift)+'_'+str(d_window)
            lag_df[col_name] = lag_df.groupby(['id'])[TARGET].transform(lambda x: x.shift(d_shift).rolling(d_window).mean()).astype(np.float16)
            
    # Drop sales column from lag_df
    del lag_df['sales']
    print('Save Lags')
    lag_df.to_pickle(processed_data_dir+f'lags_df_{SHIFT_DAY}_{store}.pkl')
    del lag_df
    
    # -------------------------------------------------------------
    # 5. MEAN ENCODINGS
    # -------------------------------------------------------------
    print('Mean Encodings')
    mean_enc_df = grid_df[['id','d','sales','state_id','store_id','cat_id','dept_id','item_id']].copy()
    
    icols =  [
                ['state_id'],
                ['store_id'],
                ['cat_id'],
                ['dept_id'],
                ['state_id', 'cat_id'],
                ['state_id', 'dept_id'],
                ['store_id', 'cat_id'],
                ['store_id', 'dept_id'],
                ['item_id'],
                ['item_id', 'state_id'],
                ['item_id', 'store_id']
                ]
                
    for col in icols:
        col_name = '_'+'_'.join(col)+'_'
        mean_enc_df['enc'+col_name+'mean'] = mean_enc_df.groupby(col)['sales'].transform('mean').astype(np.float16)
        mean_enc_df['enc'+col_name+'std'] = mean_enc_df.groupby(col)['sales'].transform('std').astype(np.float16)
        
    # keep only encoding columns and id, d
    keep_cols = [col for col in list(mean_enc_df) if col.startswith('enc')]
    mean_enc_df = mean_enc_df[['id','d']+keep_cols]
    
    print('Save Mean Encodings')
    mean_enc_df.to_pickle(processed_data_dir+f'mean_encoding_df_{store}.pkl')
    del mean_enc_df
    
    del grid_df
    print(f'Finished {store}!')

print('All 50 files generated successfully!')

